# Reset the governed project environment

This notebook is destructive. It drops the selected Unity Catalog resources so you can rebuild the project from scratch.

Use it only when you intend to remove the full project catalog and its child schemas, tables, and volumes.


In [ ]:
from __future__ import annotations

dbutils.widgets.text("catalogs", "healthcare", "Catalogs (comma-separated)")
dbutils.widgets.text("volumes", "healthcare.bronze.raw_landing", "Volumes (comma-separated)")
dbutils.widgets.text("confirm", "NO", "Type YES to continue")


def _quote_identifier(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"


def _split_csv(raw_value: str) -> list[str]:
    return [part.strip() for part in raw_value.split(",") if part.strip()]


def _normalize_catalogs(raw_value: str) -> list[str]:
    catalogs = _split_csv(raw_value)
    if not catalogs:
        raise ValueError("At least one catalog name is required.")
    return catalogs


def _normalize_volumes(raw_value: str) -> list[str]:
    volumes = _split_csv(raw_value)
    normalized: list[str] = []

    for volume in volumes:
        parts = [part.strip().strip("`") for part in volume.split(".")]
        if len(parts) != 3 or any(not part for part in parts):
            raise ValueError(
                "Each volume must be fully qualified as catalog.schema.volume. "
                f"Got: {volume!r}"
            )
        normalized.append(".".join(_quote_identifier(part) for part in parts))

    return normalized


catalogs = _normalize_catalogs(dbutils.widgets.get("catalogs"))
volumes = _normalize_volumes(dbutils.widgets.get("volumes"))
confirm = dbutils.widgets.get("confirm").strip().upper()

if confirm != "YES":
    raise ValueError("Type YES in the confirm widget before running this notebook.")

reset_sql: list[str] = []

for volume in volumes:
    reset_sql.append(f"DROP VOLUME IF EXISTS {volume}")

for catalog in catalogs:
    reset_sql.append(f"DROP CATALOG IF EXISTS {_quote_identifier(catalog)} CASCADE")

print("Planned teardown commands:")
for statement in reset_sql:
    print(f"  {statement}")


In [ ]:
for statement in reset_sql:
    print(f"Executing: {statement}")
    spark.sql(statement)

print("Reset complete.")
print("Managed tables and volumes may still be present in Databricks soft-delete retention windows.")


In [ ]:
print("Next step: run src/notebooks/bootstrap_bronze_landing.ipynb to recreate the catalog and landing volume.")
